# Jour 1 · Découper et préparer sans tricher

## Objectifs

- comprendre les rôles du train, de la validation et du test
- utiliser un découpage stratifié et reproductible
- éviter les fuites pendant la préparation


## Pourquoi trois ensembles ?

Si nous évaluons un modèle sur les mêmes exemples que ceux utilisés pour apprendre, nous mesurons surtout sa mémoire. Le **train** ajuste le modèle. La **validation** aide à comparer des choix. Le **test** reste fermé jusqu'à la décision finale et simule des données inconnues.


![Rôles du train de la validation et du test](../ressources/illustrations/jour_01/02_train_validation_test.png)

*Le test ne participe ni à l’apprentissage ni au choix du modèle.*


## Conserver la classe rare dans chaque ensemble

![Illustration du découpage stratifié](../ressources/illustrations/jour_01/02_stratification.png)

Dans ce cours, environ 60 % des lignes servent au **train**, 20 % à la **validation** et 20 % au **test**. Ces proportions ne sont pas universelles : elles doivent laisser assez d’exemples pour apprendre et assez d’exemples indépendants pour mesurer.

Comme les pannes sont rares, un tirage entièrement aléatoire pourrait créer un petit ensemble contenant trop peu de pannes. `stratify=y` demande de conserver approximativement la proportion de chaque classe.

Stratifier ne fabrique pas de nouvelles pannes et ne rend pas les classes équilibrées. Cela évite seulement qu’un ensemble reçoive une répartition très différente par hasard.


### Préparer les outils, puis ouvrir le tableau

`import` rend disponible une bibliothèque, c'est-à-dire un ensemble d'outils déjà écrits. `as pd`, par exemple, lui donne un nom court. **pandas** lit et manipule les tableaux ; **NumPy** calcule avec des tableaux de nombres ; **Matplotlib** dessine. Une **variable** est un nom associé à une valeur : `df` désignera notre tableau.

Exécutez les cellules de haut en bas, avec **Maj + Entrée**. Une cellule terminée peut produire un texte, un tableau ou un graphique ; l'absence de sortie n'est pas une erreur.


In [ ]:
from pathlib import Path  # Path décrit un chemin de fichier, sur Windows comme sur Linux.
import numpy as np  # np désigne les outils de calcul numérique.
import pandas as pd  # pd désigne les outils de tableaux.
import matplotlib.pyplot as plt  # plt désigne les outils de dessin.
from IPython.display import display  # display affiche joliment un tableau dans Jupyter.

plt.style.use("seaborn-v0_8-whitegrid")  # Ajoute une grille lisible aux graphiques.


Le petit outil suivant retrouve le dossier du cours. `def` définit une **fonction**, un bloc réutilisable ; `for` essaie plusieurs dossiers ; `if` teste une condition ; `return` fournit le résultat. `Path.cwd()` est le dossier courant et `.parents` ses dossiers parents. Le `/` entre chemins assemble les noms, il ne fait pas une division.

C'est une étape technique de chargement, pas une méthode de Machine Learning. Vous pouvez exécuter cette fonction telle quelle.


In [ ]:
def find_course_root():
    """Chercher le dossier qui contient les données du cours."""
    current = Path.cwd()  # Dossier depuis lequel Jupyter exécute ce notebook.
    for candidate in (current, *current.parents):  # * ajoute les parents à la liste d'essais.
        data_path = candidate / "ressources" / "donnees" / "maintenance_machines.csv"
        if data_path.exists():  # Arrêter la recherche dès que le fichier est trouvé.
            return candidate
    raise FileNotFoundError("Ouvrez le notebook depuis le dossier fondamentaux_ml_dl.")

ROOT = find_course_root()  # ROOT garde le chemin du dossier contenant les données.
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"


In [ ]:
df = pd.read_csv(DATA_FILE)  # Lire le CSV : df est maintenant un DataFrame (tableau).
display(df.head(3))  # Montrer 3 lignes seulement ; le tableau complet n'est pas modifié.


### Retrouver les entrées et la réponse

Une liste Python utilise des crochets : `["a", "b"]`. `df[features]` sélectionne plusieurs colonnes et garde un tableau 2D ; `df["failure_7d"]` sélectionne une colonne et produit une **Series**, une suite 1D de réponses. Les lignes de `X` et `y` doivent décrire exactement les mêmes observations, dans le même ordre.


In [ ]:
features = [  # Noms des mesures disponibles à l'instant de la décision.
    "temperature_c", "vibration_mm_s", "pressure_bar",
    "load_pct", "age_years", "days_since_maintenance",
]
X = df[features]  # Entrées : une ligne par observation, 6 colonnes par ligne.
y = df["failure_7d"]  # Réponses connues : 0 = pas de panne ; 1 = panne dans 7 jours.
print("X :", X.shape, "| y :", y.shape)  # shape donne les dimensions, pas les valeurs.


### Découper ensemble les entrées et les réponses

`train_test_split` renvoie quatre objets dans cet ordre : entrées conservées, entrées réservées, réponses conservées, réponses réservées. On écrit quatre noms à gauche de `=` pour les récupérer.

`test_size=0.40` réserve 40 % du total dans un bloc **temporaire**, pas dans le test final. `stratify=y` conserve approximativement la proportion de pannes. `random_state=42` refait le même tirage, pour que chacun puisse comparer ses résultats.


In [ ]:
from sklearn.model_selection import train_test_split  # Outil pour séparer des lignes.

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,  # Réserver 40 % ; les 60 % restants servent à apprendre.
    stratify=y,  # Conserver approximativement le taux de panne dans les deux groupes.
    random_state=42,  # Fixer le tirage pseudo-aléatoire, sans améliorer le modèle.
)


Le bloc temporaire contient 40 % du total. Le partager en deux moitiés donne **20 % de validation et 20 % de test**. Le train sert à apprendre, la validation à comparer les choix ; on réserve le test pour la décision finale.


In [ ]:
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp,  # Découper uniquement le bloc réservé précédemment.
    test_size=0.50,  # La moitié de 40 % donne 20 % du total pour le test final.
    stratify=y_temp,  # Préserver la proportion de pannes à l'intérieur de ce bloc.
    random_state=42,
)
print("Train / validation / test :", len(X_train), len(X_validation), len(X_test))


**Vérification.** Les trois nombres doivent s'additionner au nombre total de lignes. Pour ces données pédagogiques, chaque ligne concerne une machine différente. Avec plusieurs observations d'une même machine, ou pour prévoir le futur réel, il faudrait aussi séparer les machines ou le temps : une stratification seule ne suffit pas.


## Lire le code et les résultats du découpage

Le premier `train_test_split` garde 60 % des lignes dans le train et place 40 % dans `X_temp`. Le second partage ce bloc temporaire en deux moitiés : validation et test représentent donc chacun 20 % du total.

Les objets `X_train` et `y_train` doivent toujours rester alignés : la ligne d’une machine dans `X_train` correspond au label situé au même index dans `y_train`.

`random_state=42` fixe le tirage pseudo-aléatoire. Deux personnes obtiennent ainsi le même découpage et peuvent comparer leurs résultats. La valeur 42 n’a aucune propriété spéciale et n’améliore pas le modèle.

Après l’exécution, comparez les formes et les taux de panne affichés. Ils ne sont pas rigoureusement identiques, mais doivent rester proches grâce à la stratification.

## La fuite de données

Une fuite apparaît lorsqu’une information venant de la validation, du test ou du futur influence l’apprentissage. Le score obtenu paraît alors meilleur que ce que le modèle pourra réellement reproduire.


![Mauvaise et bonne préparation des données](../ressources/illustrations/jour_01/02_fuite_donnees.png)

*On découpe d’abord ; les transformations apprennent ensuite leurs paramètres uniquement sur le train.*


### Trois fuites fréquentes

- **Fuite de cible** : la réponse, ou une colonne qui la révèle presque directement, se retrouve dans `X`.
- **Fuite de préparation** : une moyenne, un minimum ou une sélection de colonnes est calculé sur tout le jeu avant le découpage.
- **Fuite temporelle** : une information connue après l’instant de décision sert à prédire le passé.

Par exemple, le nombre de jours nécessaires pour réparer une panne ne peut pas servir à prévoir cette panne : cette durée n’est connue qu’après l’événement.

La règle pratique est simple : se placer mentalement à l’instant de la prédiction et masquer tout ce qui ne serait pas encore connu.


### Vérifier aussi les groupes et le temps

Deux observations presque identiques ne deviennent pas indépendantes parce qu'on les met dans deux ensembles différents. Pour plusieurs lignes d'une même machine, on sépare les **machines** ; pour une prévision future, on place le passé dans le train et des dates plus récentes dans la validation puis le test. Le découpage aléatoire de ce cours est adapté à son fichier synthétique, pas une recette universelle.


## Pourquoi utiliser une Pipeline ?

Certaines méthodes fonctionnent mieux lorsque les variables sont standardisées. `StandardScaler` apprend la moyenne et l'écart-type de chaque colonne. Une `Pipeline` garantit que le scaler est ajusté sur le train pendant `fit`, puis simplement réutilisé pendant `predict`.

`StandardScaler` réalise une **standardisation** : pour chaque colonne, il retire la moyenne du train puis divise par son écart-type. Les variables se retrouvent ainsi sur des échelles comparables. Il mémorise les valeurs du train et réutilise exactement les mêmes sur validation et test.

La **régression logistique**, malgré son nom, est ici un modèle de classification. Elle combine les variables pour produire un score de probabilité de la classe `1`. `max_iter=1000` lui laisse assez d'étapes pour terminer son ajustement. `class_weight="balanced"` accorde automatiquement davantage d'importance à la classe rare afin que les pannes ne soient pas noyées par les nombreux cas normaux.



### Standardiser avec trois températures

![Calcul de la standardisation, puis application à une nouvelle valeur](../ressources/illustrations/jour_01/02_standardisation_calcul.png)

Pour les valeurs du train `40, 50, 60`, la moyenne vaut **50 °C**. La moyenne des écarts au carré vaut `(100 + 0 + 100) / 3` ; sa racine donne un écart-type d'environ **8,16 °C**. `StandardScaler` utilise cette division par le nombre total de valeurs.

La nouvelle valeur vaut `(température − 50) / 8,16`. C'est une distance à la moyenne exprimée en écarts-types : **0** au centre, négative en dessous, positive au-dessus. Une température de validation de **70 °C** devient **2,45** ; on utilise encore 50 et 8,16, pas une nouvelle moyenne.

`transform` renvoie un tableau NumPy. Dans `[:, 0]`, `:` sélectionne toutes les lignes et `0` la première colonne. `.round(2)` limite l'affichage à deux décimales.


In [ ]:
from sklearn.preprocessing import StandardScaler  # Outil qui apprend moyenne et écart-type.

temperatures_train = pd.DataFrame({"temperature_c": [40.0, 50.0, 60.0]})
scaler_exemple = StandardScaler()  # Créer un transformateur encore non ajusté.
scaler_exemple.fit(temperatures_train)  # Apprendre uniquement sur ces trois valeurs de train.
train_standardise = scaler_exemple.transform(temperatures_train)  # Appliquer le calcul appris.
nouvelle_temperature = pd.DataFrame({"temperature_c": [70.0]})
print("Train standardisé :", train_standardise[:, 0].round(2))  # Toutes les lignes, colonne 0.
print("Validation à 70 °C :", scaler_exemple.transform(nouvelle_temperature).round(2))


La moyenne et l'échelle apprises n'ont pas changé lorsqu'on a transformé 70 °C. Les données de validation sont exprimées dans les mêmes coordonnées que les données d'entraînement.


![Standardisation et Pipeline sans fuite](../ressources/illustrations/jour_01/02_standardisation_pipeline.png)

### `fit`, `transform` et `predict`

Un transformateur possède deux étapes distinctes. `fit` apprend ce qui est nécessaire sur le train, ici la moyenne et l’écart-type de chaque colonne. `transform` applique ensuite exactement la même transformation au train, à la validation ou au test.

La standardisation ne supprime pas les valeurs extrêmes et ne rend pas une variable « normale ». Elle recentre simplement chaque colonne autour de zéro et ramène son échelle à une variation comparable. Elle aide souvent les modèles linéaires, les méthodes basées sur des distances et les réseaux de neurones. Les arbres de décision y sont généralement moins sensibles.

Une `Pipeline` enchaîne les opérations dans le bon ordre. Lors de `fit`, elle ajuste le scaler puis le modèle sur le train. Lors de `predict`, elle transforme les nouvelles lignes avec les valeurs déjà apprises avant de les transmettre au modèle.

### Préparer ne signifie pas seulement standardiser

Selon le jeu de données, il faut aussi traiter les valeurs manquantes, les catégories, les doublons, les unités incohérentes et les valeurs impossibles. Toute opération dont le résultat dépend des données doit être apprise sur le train uniquement et, idéalement, placée dans la Pipeline.


### Mettre cette règle dans une Pipeline

`make_pipeline` assemble deux objets : d'abord le transformateur, ensuite le classifieur. `LogisticRegression` est une **classification** malgré son nom. Les parenthèses créent les objets ; `fit` les ajustera ensemble. `class_weight="balanced"` augmente l'importance des pannes rares, mais ne fabrique aucun exemple et ne garantit pas un meilleur score.


In [ ]:
from sklearn.pipeline import make_pipeline  # Enchaîner préparation et apprentissage.
from sklearn.linear_model import LogisticRegression  # Modèle de classification binaire.

pipeline = make_pipeline(
    StandardScaler(),  # Étape 1 : apprendre les échelles des features sur le train.
    LogisticRegression(
        max_iter=1000,  # Autoriser jusqu'à 1 000 étapes de calcul pour ajuster les coefficients.
        class_weight="balanced",  # Donner plus de poids à la classe rare lors de fit.
        random_state=42,
    ),
)
pipeline.fit(X_train, y_train)  # Ajuster le scaler puis le modèle, seulement avec le train.
print("Pipeline entraînée uniquement sur le train.")


## À vous de jouer — vérifier le découpage

Complétez seulement les deux valeurs marquées `None` ci-dessous pour le train :

- sa proportion est `nombre de lignes du train / nombre de lignes total` ;
- son taux de panne est la moyenne de ses réponses 0/1.

Le tableau est une liste de trois lignes, regroupée dans un `DataFrame`. Vérifiez ensuite que les proportions font 100 % et que les taux de panne restent proches.


In [ ]:
proportion_train = None  # À compléter avec len(X_train) / len(X).
taux_train = None  # À compléter avec la moyenne de y_train.
resume = pd.DataFrame({  # Chaque liste contient train, validation, puis test.
    "lignes": [len(X_train), len(X_validation), len(X_test)],
    "proportion": [proportion_train, len(X_validation) / len(X), len(X_test) / len(X)],
    "taux_panne": [taux_train, y_validation.mean(), y_test.mean()],
}, index=["train", "validation", "test"])
display(resume)  # Les cases vides restent visibles tant que le train n'est pas complété.


### Petite vérification — réutiliser la bonne moyenne

Pour une mesure de **70 °C** en validation, faut-il calculer une nouvelle moyenne incluant 70 °C ? Écrivez votre réponse et la raison en une phrase.


In [ ]:
reponse_standardisation = ""  # Réponse et raison à rédiger.
print(reponse_standardisation or "Réponse à compléter.")


**À retenir :** toute opération qui « apprend » quelque chose des données doit être ajustée sur le train seulement.
